## STEP 2

In [26]:
# -----------------------------
# Setup: Paths + sanity check
# -----------------------------

import os

PROJECT_DIR = "/content/drive/MyDrive/pphd_agent_demo"
RAG_DIR = os.path.join(PROJECT_DIR, "rag_corpus")
INDEX_DIR = os.path.join(PROJECT_DIR, "vector_store_faiss")

os.makedirs(INDEX_DIR, exist_ok=True)

print("RAG_DIR:", RAG_DIR)
print("PDF files:", [f for f in os.listdir(RAG_DIR) if f.lower().endswith(".pdf")])
print("INDEX_DIR:", INDEX_DIR)

RAG_DIR: /content/drive/MyDrive/pphd_agent_demo/rag_corpus
PDF files: ['NICE NG222 – Depression in adults- treatment and management.pdf', 'NICE CG113 – Generalised anxiety disorder and panic disorder in adults.pdf', 'WHO mhGAP Intervention Guide v2.0.pdf']
INDEX_DIR: /content/drive/MyDrive/pphd_agent_demo/vector_store_faiss


In [27]:
# -----------------------------
# Dependencies for STEP 2 (FAISS + PDF parsing + embeddings)
# -----------------------------
!pip -q install faiss-cpu pypdf sentence-transformers

In [28]:
# -----------------------------
# Build RAG corpus from PDF pages (keep page numbers for citations)
# Output: corpus = list of chunks with metadata {doc_id, page, chunk_id, text}
# -----------------------------
from pypdf import PdfReader

def load_pdf_pages(pdf_path: str):
    """Extract per-page text from a PDF."""
    reader = PdfReader(pdf_path)
    pages = []
    for i, page in enumerate(reader.pages, start=1):
        txt = page.extract_text() or ""
        txt = " ".join(txt.split())  # normalize whitespace
        if txt.strip():
            pages.append({"page": i, "text": txt})
    return pages

def chunk_text(text: str, chunk_size: int = 500, overlap: int = 100):
    """
    Simple word-based chunking.
    chunk_size/overlap are tuned for a quick demo (not perfect, but reliable).
    """
    words = text.split()
    chunks = []
    start = 0
    while start < len(words):
        end = min(len(words), start + chunk_size)
        chunk = " ".join(words[start:end])
        chunks.append(chunk)
        if end == len(words):
            break
        start = max(0, end - overlap)
    return chunks

def build_corpus(rag_dir: str):
    """Read all PDFs in rag_dir, split into chunks, attach citation metadata."""
    corpus = []
    for fn in os.listdir(rag_dir):
        if not fn.lower().endswith(".pdf"):
            continue
        path = os.path.join(rag_dir, fn)

        pages = load_pdf_pages(path)
        for p in pages:
            chunks = chunk_text(p["text"], chunk_size=500, overlap=100)
            for j, ch in enumerate(chunks):
                corpus.append({
                    "doc_id": fn,
                    "page": p["page"],
                    "chunk_id": j,
                    "text": ch
                })
    return corpus

corpus = build_corpus(RAG_DIR)
print("Total chunks:", len(corpus))
print("Sample chunk meta:", corpus[0]["doc_id"], "page", corpus[0]["page"], "chunk", corpus[0]["chunk_id"])
print("Sample text:", corpus[0]["text"][:200], "...")

Total chunks: 344
Sample chunk meta: NICE NG222 – Depression in adults- treatment and management.pdf page 1 chunk 0
Sample text: Depression in adults: treatment and management NICE guideline Published: 29 June 2022 www.nice.org.uk/guidance/ng222 © NICE 2026. All rights reserved. Subject to Notice of rights (https://www.nice.org ...


In [29]:
# -----------------------------
# Build embeddings + FAISS index
# - We normalize embeddings and use inner product (IP) index => cosine similarity
# - Save index + metadata to Drive for reuse
# -----------------------------
import json
import numpy as np
import faiss
from sentence_transformers import SentenceTransformer

EMB_MODEL_NAME = "sentence-transformers/all-MiniLM-L6-v2"
embedder = SentenceTransformer(EMB_MODEL_NAME)

texts = [c["text"] for c in corpus]

# Encode chunks -> (N, dim) float32 embeddings
emb = embedder.encode(
    texts,
    batch_size=64,
    show_progress_bar=True,
    normalize_embeddings=True
)
emb = np.asarray(emb, dtype="float32")

dim = emb.shape[1]

# FAISS index for cosine similarity (with normalized embeddings)
index = faiss.IndexFlatIP(dim)
index.add(emb)

# Persist index and metadata
faiss.write_index(index, os.path.join(INDEX_DIR, "index.faiss"))
with open(os.path.join(INDEX_DIR, "meta.json"), "w", encoding="utf-8") as f:
    json.dump(corpus, f)

# Also store model name for reproducibility
with open(os.path.join(INDEX_DIR, "embedder.txt"), "w", encoding="utf-8") as f:
    f.write(EMB_MODEL_NAME)

print("Saved FAISS index ->", os.path.join(INDEX_DIR, "index.faiss"))
print("Saved metadata   ->", os.path.join(INDEX_DIR, "meta.json"))
print("Embedder model   ->", EMB_MODEL_NAME)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Batches:   0%|          | 0/6 [00:00<?, ?it/s]

Saved FAISS index -> /content/drive/MyDrive/pphd_agent_demo/vector_store_faiss/index.faiss
Saved metadata   -> /content/drive/MyDrive/pphd_agent_demo/vector_store_faiss/meta.json
Embedder model   -> sentence-transformers/all-MiniLM-L6-v2


In [30]:
# -----------------------------
# Retrieval API for downstream agent
# Input: slots (from Day1 payload)
# Output: top_k evidence chunks with doc_id/page/snippet/score
# -----------------------------
import json
import numpy as np
import faiss

# Reload (robust to runtime restart)
index = faiss.read_index(os.path.join(INDEX_DIR, "index.faiss"))
with open(os.path.join(INDEX_DIR, "meta.json"), "r", encoding="utf-8") as f:
    meta = json.load(f)

def slots_to_query(slots: dict) -> str:
    """
    Query template tuned for guideline retrieval:
    - prefer actionable sections (treatment/management/first-line/CBT/self-help)
    - avoid generic terms that lead to meta pages (research/committee info)
    """
    symptoms = slots.get("symptoms") or []
    duration = slots.get("duration") or ""

    # Keep the top 1-2 symptoms to reduce topic drift
    core = " ".join(symptoms[:2])

    # Add action-oriented anchors to bias retrieval toward recommendations
    anchors = "treatment management first-line stepped care CBT self-help referral"

    return f"{core} {duration} {anchors}".strip()

# -----------------------------
# Post-filter retrieved chunks to avoid meta sections
# (research recommendations, committee/admin pages, change logs, etc.)
# -----------------------------
BAD_PATTERNS = [
    "recommendations for research",
    "committee",
    "guideline development",
    "updat",          # matches update/updated
    "why this is important",
    "methods",
    "appendix",
    "references",
    "glossary",
    "acknowledg",
    "introduction"
]

BAD_PATTERNS += [
    "overview",
    "replaces",
    "basis of",
    "this guideline",
    "updating",
    "scope"
]

def is_bad_chunk(snippet: str) -> bool:
    s = snippet.lower()
    return any(p in s for p in BAD_PATTERNS)

# -----------------------------
# Retrieval with (1) bad-chunk filtering and (2) simple page diversity
# -----------------------------
def retrieve(slots: dict, top_k: int = 5, oversample: int = 40):
    """
    Retrieve top_k evidence chunks with:
    - oversampling from FAISS
    - filtering out obvious non-recommendation sections
    - enforcing page diversity (avoid returning 5 chunks from the same page)
    """
    query = slots_to_query(slots)
    q_emb = embedder.encode([query], normalize_embeddings=True)
    q_emb = np.asarray(q_emb, dtype="float32")

    # Oversample then filter down
    scores, idxs = index.search(q_emb, oversample)

    results = []
    seen_pages = set()

    for score, i in zip(scores[0], idxs[0]):
        item = meta[int(i)]
        snippet = item["text"]

        # Skip meta / non-actionable sections
        if is_bad_chunk(snippet):
            continue

        # Simple diversity: avoid repeating the same (doc_id, page)
        page_key = (item["doc_id"], item["page"])
        if page_key in seen_pages:
            continue
        seen_pages.add(page_key)

        results.append({
            "score": float(score),
            "doc_id": item["doc_id"],
            "page": item["page"],
            "chunk_id": item["chunk_id"],
            "snippet": snippet[:600]
        })

        if len(results) >= top_k:
            break

    return query, results

In [31]:
PAYLOAD_PATH = os.path.join(PROJECT_DIR, "sample_payload.json")

with open(PAYLOAD_PATH, "r", encoding="utf-8") as f:
    payload = json.load(f)

query, hits = retrieve(payload["slots"], top_k=5)

In [32]:
# -----------------------------
# Sanity test: retrieve evidence using the Day1 payload slots
# -----------------------------
query, hits = retrieve(payload["slots"], top_k=5)
print("QUERY:", query)
for h in hits:
    print(f"- score={h['score']:.3f} | {h['doc_id']} p{h['page']} | {h['snippet'][:140]}...")

QUERY: anxiety sleep_disturbance 2_week treatment management first-line stepped care CBT self-help referral
- score=0.658 | NICE CG113 – Generalised anxiety disorder and panic disorder in adults.pdf p21 | on all aspects of anxiety disorders plus other sources of help.) [2004] 1.3.11 The benefits of exercise as part of good general health shoul...
- score=0.607 | NICE CG113 – Generalised anxiety disorder and panic disorder in adults.pdf p41 | mental health problems. Recommendations in section 1.2 on stepped care for people with panic disorder were reordered. Recommendations marked...
- score=0.606 | NICE CG113 – Generalised anxiety disorder and panic disorder in adults.pdf p27 | Monitoring and follow-up for individuals with panic disorder Psychological interventions 1.3.40 There should be a process within each practi...
- score=0.574 | NICE CG113 – Generalised anxiety disorder and panic disorder in adults.pdf p20 | • undergo the minimum investigations necessary to exclude acute physical

## STEP 3

In [33]:
# -----------------------------
# STEP 3: Build a copy-paste prompt for ChatGPT (Consultant Agent)
# -----------------------------
def format_3_prompt(slots: dict, hits: list) -> str:
    evidence_lines = []
    for i, h in enumerate(hits, start=1):
        snippet = h["snippet"].replace("\n", " ").strip()
        evidence_lines.append(f"[{i}] {h['doc_id']} p{h['page']}\n{snippet}")

    evidence_block = "\n\n".join(evidence_lines)

    instructions = """
Prompt (Consultant Agent):

You are a clinical guideline assistant. Use ONLY the evidence provided.

Task:
1) Write 3–5 actionable, non-diagnostic recommendations for the user.
2) After EACH recommendation, add a citation in the exact format: (doc_id, page).
3) Do not invent facts or recommendations not supported by the evidence.
4) Use cautious language (“may”, “consider”, “it can help to…”) and avoid prescribing medication.
5) If safety_flags include self-harm risk, add a brief safety note encouraging immediate professional help and crisis resources.

Output format:
• Recommendation 1 … (doc_id, page)
• Recommendation 2 … (doc_id, page)
...
""".strip()

    return f"""=== COPY THIS INTO CHATGPT (Day3) ===

User slots:
- symptoms: {slots.get('symptoms')}
- duration: {slots.get('duration')}
- safety_flags: {slots.get('safety_flags')}

Evidence (doc_id + page):
{evidence_block}

{instructions}
"""

STEP3_prompt = format_3_prompt(payload["slots"], hits)
print(STEP3_prompt)

=== COPY THIS INTO CHATGPT (Day3) ===

User slots:
- symptoms: ['anxiety', 'sleep_disturbance', 'depression']
- duration: 2_week
- safety_flags: []

Evidence (doc_id + page):
[1] NICE CG113 – Generalised anxiety disorder and panic disorder in adults.pdf p21
on all aspects of anxiety disorders plus other sources of help.) [2004] 1.3.11 The benefits of exercise as part of good general health should be discussed with all people with panic disorder as appropriate. [2004] Step 3 for people with panic disorder: review and offer alternative treatment if appropriate 1.3.12 For people with moderate to severe panic disorder (with or without agoraphobia), consider referral for: • CBT or • an antidepressant if the disorder is long-standing or the person has not benefitted from or has declined psychological intervention. [2011] Psychological interventions 1.3

[2] NICE CG113 – Generalised anxiety disorder and panic disorder in adults.pdf p41
mental health problems. Recommendations in section 1.2 on

In [34]:
# -----------------------------
# Paste ChatGPT output here (manual step)
# -----------------------------
advice_text = """
• Consider discussing the role of regular exercise as part of overall health support, since guideline text notes that the benefits of exercise as part of good general health should be discussed where appropriate. (NICE CG113 – Generalised anxiety disorder and panic disorder in adults.pdf, p21)

• If symptoms feel moderate-to-severe or are not improving, consider seeking a clinical assessment and discussing a referral for evidence-based psychological therapy such as CBT, which is explicitly listed as a treatment option to consider in the guideline pathway. (NICE CG113 – Generalised anxiety disorder and panic disorder in adults.pdf, p21)

• If you start CBT, consider setting up a structured process with your provider/practice to track progress and adjust the plan as needed, as the guideline recommends having a process to assess progress during CBT on a case-by-case basis. (NICE CG113 – Generalised anxiety disorder and panic disorder in adults.pdf, p27)

• If you and a clinician decide to start any new medication, consider planning follow-up reviews early (for example, around 2 weeks after initiation and again at later intervals), because the guideline specifies reviewing efficacy and side effects within 2 weeks of starting a new medication and again at 4, 6, and 12 weeks. (NICE CG113 – Generalised anxiety disorder and panic disorder in adults.pdf, p27)

• Consider asking for (or using) written information about panic attacks and reasons for care pathways, and exploring local/national support resources (including voluntary and self-help groups), as the guideline recommends offering written information and support-source information as part of subsequent care. (NICE CG113 – Generalised anxiety disorder and panic disorder in adults.pdf, p20)
""".strip()

print(advice_text)

• Consider discussing the role of regular exercise as part of overall health support, since guideline text notes that the benefits of exercise as part of good general health should be discussed where appropriate. (NICE CG113 – Generalised anxiety disorder and panic disorder in adults.pdf, p21)

• If symptoms feel moderate-to-severe or are not improving, consider seeking a clinical assessment and discussing a referral for evidence-based psychological therapy such as CBT, which is explicitly listed as a treatment option to consider in the guideline pathway. (NICE CG113 – Generalised anxiety disorder and panic disorder in adults.pdf, p21)

• If you start CBT, consider setting up a structured process with your provider/practice to track progress and adjust the plan as needed, as the guideline recommends having a process to assess progress during CBT on a case-by-case basis. (NICE CG113 – Generalised anxiety disorder and panic disorder in adults.pdf, p27)

• If you and a clinician decide to

In [35]:
# -----------------------------
# Simple verifier (each bullet must include a (doc_id, page) citation)
# -----------------------------
import re

CIT_RE = re.compile(r"\([^)]*\.pdf,\s*p\d+\)", re.I)

def verify_citations(advice: str) -> bool:
    lines = [ln.strip() for ln in advice.splitlines() if ln.strip()]
    rec_lines = [ln for ln in lines if ln.startswith("•")]
    if not rec_lines:
        return False
    return all(CIT_RE.search(ln) for ln in rec_lines)

print("citation_check_pass:", verify_citations(advice_text))

citation_check_pass: True


In [37]:
# -----------------------------
# Save final answer to Drive (project artifact)
# -----------------------------
import os

OUT_DIR = os.path.join(PROJECT_DIR, "outputs")
os.makedirs(OUT_DIR, exist_ok=True)

OUT_PATH = os.path.join(OUT_DIR, "step3_advice.md")
with open(OUT_PATH, "w", encoding="utf-8") as f:
    f.write(advice_text + "\n")

print("saved ->", OUT_PATH)

saved -> /content/drive/MyDrive/pphd_agent_demo/outputs/step3_advice.md
